In [1]:
import pandas as pd
import numpy as np
import sweetviz as sv

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    StratifiedKFold
)

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from xgboost import XGBClassifier

In [2]:
# importing data
train = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\train.csv")
test = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\test.csv")
sample = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\sample_submission.csv")

train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
report = sv.analyze(train)
report.show_html("irrigation_eda_report.html")

                                             |          | [  0%]   00:00 -> (? left)

Report irrigation_eda_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [4]:
# quick checks
print(train.shape)
print(test.shape)
print(train.isna().sum())
print(train.duplicated().sum())
print(train["Irrigation_Need"].value_counts())
print(train["Irrigation_Need"].value_counts(normalize=True))

(630000, 21)
(270000, 20)
id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64
0
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64
Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: proportion, dtype: float64


In [5]:
# preprocessing for random forest
rf_train = train.copy()
rf_test = test.copy()

le = LabelEncoder()
y = le.fit_transform(rf_train["Irrigation_Need"])

X = rf_train.drop(["Irrigation_Need"], axis=1)
X_test_final = rf_test.copy()

full = pd.concat([X, X_test_final], axis=0)
full_encoded = pd.get_dummies(full, drop_first=False)

X = full_encoded.iloc[:len(X), :]
X_test_final = full_encoded.iloc[len(X):, :]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# baseline random forest
rf_base = RandomForestClassifier(
    random_state=42,
    n_estimators=200,
    class_weight="balanced",
    n_jobs=-1
)

rf_cv_scores = cross_val_score(
    rf_base,
    X, y,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

print("Random Forest CV balanced accuracy scores:", rf_cv_scores)
print("Mean CV balanced accuracy:", rf_cv_scores.mean())

Random Forest CV balanced accuracy scores: [0.95407597 0.95474065 0.95461565 0.95566362 0.95221471]
Mean CV balanced accuracy: 0.9542621188008831


In [6]:
# tuning random forest
param_grid_rf = {
    "n_estimators": [200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced"]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("Best RF parameters:", grid_rf.best_params_)
print("Best RF CV balanced accuracy:", grid_rf.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best RF parameters: {'class_weight': 'balanced', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 300}
Best RF CV balanced accuracy: 0.9645676266591428


In [7]:
# final rf validation results
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_valid)

print("RF validation balanced accuracy:", balanced_accuracy_score(y_valid, y_pred_rf))
print("RF validation accuracy:", accuracy_score(y_valid, y_pred_rf))
print("RF validation macro F1:", f1_score(y_valid, y_pred_rf, average="macro"))
print(classification_report(y_valid, y_pred_rf, target_names=le.classes_))

RF validation balanced accuracy: 0.9668628482326517
RF validation accuracy: 0.9847539682539682
RF validation macro F1: 0.9658620118506055
              precision    recall  f1-score   support

        High       0.92      0.93      0.93      4202
         Low       0.99      1.00      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.96      0.97      0.97    126000
weighted avg       0.98      0.98      0.98    126000



In [8]:
# kaggle submission
rf_test_pred = best_rf.predict(X_test_final)
rf_test_labels = le.inverse_transform(rf_test_pred)

rf_submission = sample.copy()
rf_submission["Irrigation_Need"] = rf_test_labels
rf_submission.to_csv("rf_submission.csv", index=False)
rf_submission.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [9]:
# preprocessing for xgboost
xgb_train = train.copy()
xgb_test = test.copy()

le = LabelEncoder()
xgb_train["Irrigation_Need"] = le.fit_transform(xgb_train["Irrigation_Need"])

X = xgb_train.drop(["Irrigation_Need"], axis=1)
y = xgb_train["Irrigation_Need"]

for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category")
    xgb_test[col] = xgb_test[col].astype("category")

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# baseline xgboost
xgb_base = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric="mlogloss"
)

xgb_cv_scores = cross_val_score(
    xgb_base,
    X, y,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

print("XGBoost CV balanced accuracy scores:", xgb_cv_scores)
print("Mean CV balanced accuracy:", xgb_cv_scores.mean())

XGBoost CV balanced accuracy scores: [0.96175838 0.96295345 0.96175625 0.96208377 0.961336  ]
Mean CV balanced accuracy: 0.9619775697468433


In [11]:
# tuning xgboost

param_grid_xgb = {
    "n_estimators": [100, 150],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8]
}


grid_xgb = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        enable_categorical=True,
        eval_metric="mlogloss"
    ),
    param_grid=param_grid_xgb,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

print("Best XGB parameters:", grid_xgb.best_params_)
print("Best XGB CV balanced accuracy:", grid_xgb.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best XGB parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 150, 'subsample': 1.0}
Best XGB CV balanced accuracy: 0.9614014633788812


In [12]:
# final xgb validation results
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_valid)

print("XGB validation balanced accuracy:", balanced_accuracy_score(y_valid, y_pred_xgb))
print("XGB validation accuracy:", accuracy_score(y_valid, y_pred_xgb))
print("XGB validation macro F1:", f1_score(y_valid, y_pred_xgb, average="macro"))
print(classification_report(y_valid, y_pred_xgb, target_names=le.classes_))

XGB validation balanced accuracy: 0.9646482558940105
XGB validation accuracy: 0.9852460317460318
XGB validation macro F1: 0.9714783845835712
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [13]:
# kaggle submission
xgb_test_pred = best_xgb.predict(xgb_test)
xgb_test_labels = le.inverse_transform(xgb_test_pred)

xgb_submission = sample.copy()
xgb_submission["Irrigation_Need"] = xgb_test_labels
xgb_submission.to_csv("xgb_submission.csv", index=False)
xgb_submission.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687104
- While I did look at this discussion and found it useful for knowing the feature importance, XGBoost seems to handle feature reduduancy well. However, I will be keeping in mind these features for my further phases of the kaggle competition.

https://www.kaggle.com/competitions/playground-series-s6e4/discussion/689873
- I used this when tuning my models, choosing the ranges 4 - 6 (as seen in my XGBoost Model) to find the best performance balance. 

https://www.kaggle.com/competitions/playground-series-s6e4/discussion/686709
- This boost furhter reinforced the idea that I should use balance accuracy as my metric, since the data set was so imbalanced. 


For my bagging model, I used Random Forest. The baseline model already performed very well, with a mean CV balanced accuracy of 0.954. 
After tuning, the best Random Forest used n_estimators = 300, max_depth = 10, max_features = "sqrt", min_samples_split = 5, and class_weight = "balanced". 
This improved the CV balanced accuracy to 0.965, and on the validation set it reached a balanced accuracy of 0.967 and a macro F1 of 0.966.

For my boosting model, I used XGBoost. The baseline XGBoost model performed slightly better than the Random Forest baseline, with a mean CV balanced accuracy of 0.962. 
After tuning, the best XGBoost used n_estimators = 150, max_depth = 6, learning_rate = 0.1, subsample = 1.0, and colsample_bytree = 0.8. 
The tuned CV balanced accuracy was 0.961, and the validation balanced accuracy was 0.965 with a macro F1 of 0.971.

Overall, both models performed extremely well and the differences were small. Random Forest had a slightly stronger validation balanced accuracy, while XGBoost had the higher macro F1 score. 
For my work, bagging and boosting were both effective, but XGBoost seemed a little stronger at balancing performance across classes, while Random Forest was simpler and still highly competitive.

In the future, one how improvement I will make for my models is to reduce that amount of time it takes for them to run. For efficiency sake, I should not run too many combinations, as my initial RF tuning took 50 minutes, while my XGB tuning took 5 minutes, and outputed similar results(RF = 0.954 and XGB = 0.962). The total change in model performance was not too significant . Despite the model performance showing that XGBoost was a better model than RF, my Kaggle submission showed that the tuned RF (0.96211) was slighlty better than XGBoost (0.95925) by 0.00286.

For Phasse 2, I would like to improve the feature engineering and continue fine tuning the parameteres. I could also look at other models and see if there can be any improvement with new models, such as CatBoost